# The Laplace Transform
**Category: Analysis**

## 1. Introduction

The Laplace transform is an integral transform that converts a function of a real variable $t \geq 0$ into a function of a complex variable $s$. It is one of the central tools in the analysis of linear differential equations and dynamical systems, and bears a deep structural relationship to the Fourier transform.

## 2. Definition and Basic Properties

### 2.1 Definition

Let $f : [0, \infty) \to \mathbb{C}$ be a locally integrable function. The **Laplace transform** of $f$ is defined by
$$
\mathcal{L}\{f\}(s) = F(s) = \int_0^{\infty} f(t)\, e^{-st}\, dt
$$
for all $s \in \mathbb{C}$ for which the integral converges absolutely. The variable $s = \sigma + i\omega$ is complex, with $\sigma = \operatorname{Re}(s)$.

### 2.2 Region of Convergence

If $f$ is of **exponential order** $\alpha$, meaning there exist constants $M, T > 0$ such that $|f(t)| \leq M e^{\alpha t}$ for all $t > T$, then $F(s)$ converges absolutely for all $s$ with $\operatorname{Re}(s) > \alpha$. The **abscissa of convergence** is
$$
\sigma_c = \inf \bigl\{ \sigma \in \mathbb{R} : F(\sigma + i\omega) \text{ converges for all } \omega \bigr\}
$$
and $F$ is analytic on the half-plane $\{\operatorname{Re}(s) > \sigma_c\}$.

### 2.3 Elementary Transforms

The following table collects the most frequently used pairs. Throughout, $n \in \mathbb{N}$, $a \in \mathbb{C}$, and all transforms hold for $\operatorname{Re}(s)$ sufficiently large.

| $f(t)$ | $F(s)$ |
|---|---|
| $1$ | $\dfrac{1}{s}$ |
| $e^{at}$ | $\dfrac{1}{s - a}$ |
| $t^n$ | $\dfrac{n!}{s^{n+1}}$ |
| $\sin(\omega t)$ | $\dfrac{\omega}{s^2 + \omega^2}$ |
| $\cos(\omega t)$ | $\dfrac{s}{s^2 + \omega^2}$ |

## 3. Operational Properties

### 3.1 Linearity

For $\alpha, \beta \in \mathbb{C}$ and functions $f, g$ of exponential order,
$$
\mathcal{L}\{\alpha f + \beta g\}(s) = \alpha F(s) + \beta G(s)
$$
This follows directly from linearity of the integral.

### 3.2 Differentiation in $t$

If $f$ is absolutely continuous and $f' \in L^1_{\mathrm{loc}}$, integration by parts gives
$$
\mathcal{L}\{f'\}(s) = s F(s) - f(0^+)
$$
Applying this inductively,
$$
\mathcal{L}\{f^{(n)}\}(s) = s^n F(s) - \sum_{k=0}^{n-1} s^{n-1-k}\, f^{(k)}(0^+)
$$
This formula is the principal reason the Laplace transform converts constant-coefficient ODEs into algebraic equations in $s$.

### 3.3 Convolution Theorem

The one-sided convolution of $f$ and $g$ is
$$
(f * g)(t) = \int_0^t f(\tau)\, g(t - \tau)\, d\tau
$$
The convolution theorem states
$$
\mathcal{L}\{f * g\}(s) = F(s)\, G(s)
$$
so convolution in the time domain corresponds to pointwise multiplication in the $s$-domain.

## 4. Inversion

### 4.1 Bromwich Integral

Under suitable conditions, the inverse Laplace transform is given by the **Bromwich integral** (a contour integral along a vertical line in the complex plane),
$$
f(t) = \mathcal{L}^{-1}\{F\}(t) = \frac{1}{2\pi i} \lim_{T \to \infty} \int_{\sigma - iT}^{\sigma + iT} F(s)\, e^{st}\, ds
$$
where $\sigma > \sigma_c$. In practice, the integral is evaluated by closing the contour in the left half-plane and applying the **residue theorem**:
$$
f(t) = \sum_{k} \operatorname{Res}_{s = s_k} \bigl[ F(s)\, e^{st} \bigr]
$$
where the sum runs over all poles $s_k$ of $F$.

### 4.2 Partial Fraction Decomposition

When $F(s) = P(s)/Q(s)$ is rational with $\deg P < \deg Q$ and $Q$ has simple roots $s_1, \ldots, s_m$, the decomposition
$$
F(s) = \sum_{k=1}^{m} \frac{A_k}{s - s_k}, \qquad A_k = \lim_{s \to s_k} (s - s_k)\, F(s)
$$
yields immediately
$$
f(t) = \sum_{k=1}^{m} A_k\, e^{s_k t}
$$
For a repeated pole of order $r$ at $s_0$, the contribution is $\displaystyle\sum_{j=0}^{r-1} \frac{B_j}{j!}\, t^j e^{s_0 t}$.

## 5. Numerical Experiment

### 5.1 Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad

# Numerical Laplace transform via direct quadrature
def laplace_numerical(f, s, T=40.0):
    real_part, _ = quad(lambda t: np.real(f(t) * np.exp(-s * t)), 0, T)
    imag_part, _ = quad(lambda t: np.imag(f(t) * np.exp(-s * t)), 0, T)
    return complex(real_part, imag_part)

# Stehfest algorithm for numerical inversion
def stehfest_inversion(F, t, N=16):
    ln2_t = np.log(2) / t
    n = N // 2
    V = np.zeros(N)
    for i in range(1, N + 1):
        for k in range(int((i + 1) / 2), min(i, n) + 1):
            V[i-1] += (k**n * np.math.factorial(2*k) /
                       (np.math.factorial(n - k) * np.math.factorial(k) *
                        np.math.factorial(k - 1) * np.math.factorial(i - k) *
                        np.math.factorial(2*k - i)))
        V[i-1] *= (-1)**(i + n)
    return ln2_t * sum(V[i-1] * F(ln2_t * i) for i in range(1, N + 1))

### 5.2 Exponential Decay

Consider $f(t) = e^{-at}$ for $a > 0$. The exact transform is $F(s) = (s+a)^{-1}$ for $\operatorname{Re}(s) > -a$. We verify numerically that the magnitude $|F(\sigma + i\omega)|$ decays as $\omega \to \infty$ along the line $\sigma = 1$:
$$
|F(1 + i\omega)| = \frac{1}{\sqrt{(1+a)^2 + \omega^2}} \sim \frac{1}{|\omega|} \quad \text{as } |\omega| \to \infty
$$
This $O(|\omega|^{-1})$ decay reflects the $C^\infty$ regularity of $f$ on $(0,\infty)$ and the single simple pole at $s = -a$.

In [ ]:
a = 2.0
f = lambda t: np.exp(-a * t)
F_exact = lambda s: 1.0 / (s + a)

omega = np.linspace(0.5, 30, 500)
sigma = 1.0

F_num = np.array([abs(laplace_numerical(f, sigma + 1j*w)) for w in omega])
F_ana = np.array([abs(F_exact(sigma + 1j*w)) for w in omega])

fig, ax = plt.subplots(figsize=(7, 4))
ax.loglog(omega, F_ana, label=r'$|F(1+i\omega)|$ exact', linewidth=2)
ax.loglog(omega, F_num, '--', label='numerical quadrature', linewidth=1.5)
ax.loglog(omega, 1/omega, ':', label=r'$O(\omega^{-1})$', color='gray')
ax.set_xlabel(r'$\omega$')
ax.set_ylabel(r'$|F(1+i\omega)|$')
ax.set_title('Decay of the Laplace Transform along $\\sigma = 1$')
ax.legend()
plt.tight_layout()
plt.show()

### 5.3 Inversion via Stehfest Algorithm

The Stehfest algorithm approximates the Bromwich integral using $N$ real evaluations of $F$ on the positive real axis. For $f(t) = e^{-at}$, we expect pointwise recovery. The error at time $t$ satisfies
$$
\bigl| \hat{f}_N(t) - f(t) \bigr| \to 0 \quad \text{as } N \to \infty
$$
provided $f$ is smooth, with convergence rate governed by the smoothness of $f$ and the choice of $N$.

In [ ]:
t_vals = np.linspace(0.05, 4.0, 200)
f_exact = np.exp(-a * t_vals)
f_stehfest = np.array([stehfest_inversion(F_exact, t) for t in t_vals])

fig, axes = plt.subplots(1, 2, figsize=(11, 4))

axes[0].plot(t_vals, f_exact, label='$e^{-2t}$', linewidth=2)
axes[0].plot(t_vals, f_stehfest, '--', label='Stehfest ($N=16$)', linewidth=1.5)
axes[0].set_xlabel('$t$')
axes[0].set_ylabel('$f(t)$')
axes[0].set_title('Inversion: exact vs. Stehfest')
axes[0].legend()

axes[1].semilogy(t_vals, np.abs(f_stehfest - f_exact) + 1e-16)
axes[1].set_xlabel('$t$')
axes[1].set_ylabel('absolute error')
axes[1].set_title('Pointwise inversion error')

plt.tight_layout()
plt.show()